# Baseline: Caser

Caser (Tang & Wang, 2018) con el modelo de [graytowne/caser_pytorch](https://github.com/graytowne/caser_pytorch) (commit `c4e10e4`, sin modificarlo), entrenado con el protocolo de LatentR3: Adam, grilla de lr × weight decay, BCE con negativos aleatorios y contexto de hasta 10 ítems. Cada fila de `train` es una muestra, y se reportan HR y NDCG @5 y @10 por id y por título (el criterio por título es el comparable con la Tabla 1).

Además de BCE (paper) hay tres variantes de referencia: BCE sin usuario, CE catálogo y BCE T=3. El entrenamiento está en `caser_baseline.py` y el protocolo común en `../baseline_comun.py`. El notebook verifica los datos, hace una prueba rápida, corre la grilla (una celda por variante) y arma las tablas; no escribe archivos.

Las decisiones, los costos y los resultados están en [README.md](README.md). En local lee los datos de `../../repo_latentr3/data` y el código de `../../caser_pytorch`; en Colab, de `data_amazon_2018/` y `caser_pytorch/` (con `caser_baseline.py` y `baseline_comun.py` en el directorio de trabajo).

## Importaciones

In [1]:
import os
import time
import itertools
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import torch

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


In [2]:
# La grilla corre en 6 procesos (decisión del proyecto). Costos y reproducibilidad: README.md,
# secciones "Costos medidos" y "Problemas conocidos"
N_PROCESOS = 6
N_PROCESOS_CE = 6
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

cuda AMD Radeon RX 9070 XT


## Datos y código original

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [4]:
import sys
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3', 'data')
codigo = os.path.abspath('caser_pytorch' if IN_COLAB else os.path.join('..', '..', 'caser_pytorch'))
# baseline_comun.py está en notebooks/ en local y en el directorio de trabajo en Colab. Los procesos de
# la grilla heredan sys.path, así que también lo encuentran
comun = os.path.abspath('.' if IN_COLAB else '..')
if comun not in sys.path:
    sys.path.insert(0, comun)
names = {'CDs_and_Vinyl': 'cd_data',
         'Musical_Instruments': 'music_data',
         'Toys_and_Games': 'toys_data',
         'Video_Games': 'games_data'}

for archivo in ['caser.py', 'utils.py']:
    if not os.path.isfile(os.path.join(codigo, archivo)):
        raise FileNotFoundError(f'No se encontró {archivo} en {codigo}. Descargar '
                                'https://github.com/graytowne/caser_pytorch en esa carpeta y hacer checkout del commit c4e10e4.')
for modulo in ['caser_baseline.py', os.path.join(comun, 'baseline_comun.py')]:
    if not os.path.isfile(modulo):
        raise FileNotFoundError(f'No se encontró {os.path.abspath(modulo)}.')

import caser_baseline as cb
cb.inicializar_proceso(codigo)   # verifica el sha256 de caser.py y utils.py antes de importarlos

In [5]:
resumen = {}
for categoria, nombre in names.items():
    datos, titulos, usuarios, prep = cb.cargar_categoria(base, categoria)
    n_items = len(titulos)
    for split, df in datos.items():
        S, u, y, Y3 = (prep[split][k].cpu().numpy() for k in ['S', 'u', 'y', 'Y3'])
        largos = df['history_item_id'].map(len).to_numpy()
        # Ninguna secuencia de más de L = 10 ítems, y cada fila conserva su historial completo, en orden y a la derecha
        assert S.shape == (len(df), cb.MAX_LEN) and largos.max() <= cb.MAX_LEN
        assert ((S != 0).sum(1) == largos).all()
        assert all(S[i, cb.MAX_LEN - len(h):].tolist() == [j + 1 for j in h]
                   for i, h in enumerate(df['history_item_id']))
        # 0 es el padding; dentro del modelo los ids van de 1 a n_items
        assert S.min() >= 0 and S.max() <= n_items and y.min() >= 0 and y.max() < n_items
        # Usuarios con filas en train: índices 1..U. Los demás: índice 0 (vector cero)
        conocido = df['user_id'].isin(usuarios).to_numpy()
        assert ((u[:, 0] > 0) == conocido).all() and u.max() <= len(usuarios)
        assert split != 'train' or conocido.all()
        # Y3 (variante T=3): el objetivo de la fila y los de las 2 filas siguientes del mismo usuario
        assert (Y3[:, 0] == y + 1).all()
        if split == 'train':
            for filas in df.groupby('user_id', sort=False).indices.values():
                # Las filas de un usuario son posiciones consecutivas de su secuencia: el historial de
                # cada fila termina en el objetivo de la anterior, así que no hay fuga desde valid o test
                assert all(df['history_item_id'].iat[b][-1] == y[a] for a, b in zip(filas[:-1], filas[1:]))
                for k in range(cb.T_REFERENCIA):
                    assert Y3[filas, k].tolist() == list(y[filas[k:]] + 1) + [0] * min(k, len(filas))
    # Cada fila de train aparece una vez por época
    tamanos = [len(lote['y']) for lote in cb.lotes(prep['train'], cb.BATCH_SIZE, np.random.default_rng(0))]
    assert sum(tamanos) == len(datos['train'])
    resumen[nombre] = {**{split: len(df) for split, df in datos.items()}, 'ítems': n_items,
                       'usuarios en train': len(usuarios),
                       **{f'% filas de usuarios nuevos en {s}': 100 * (prep[s]['u'] == 0).float().mean().item()
                          for s in ['valid', 'test']}}
pd.DataFrame(resumen).round(1)

,cd_data,music_data,toys_data,games_data
train,49251.0,66500.0,53898.0,75175.0
valid,6156.0,8312.0,6737.0,9397.0
test,6157.0,8313.0,6738.0,9397.0
ítems,5841.0,5030.0,6299.0,5308.0
usuarios en train,6865.0,10899.0,10167.0,12119.0
% filas de usuarios nuevos en valid,38.8,34.2,47.2,43.2
% filas de usuarios nuevos en test,50.6,43.1,43.6,51.5


## Métricas

`hit`, `ndcg`, `dedup_titulos` y `evaluar` están en `../baseline_comun.py`: HR y NDCG @5 y @10 **por id** y **por título** (como `calc.py`), con un solo ítem relevante. `evaluar` verifica que las métricas por título nunca sean menores que por id. El criterio por título se explica en el README de `notebooks/`.

`caser_baseline.py` agrega `metricas_por_usuario`: HR@10 y NDCG@10 por título, por separado para las filas de test de usuarios con filas en `train` (conocidos) y sin ellas (nuevos).

## Modelo Caser

**No lo especifica el paper:** L = 10, dimensión 50, 4 filtros verticales, 16 filtros horizontales por altura, ReLU, dropout 0.5 y batch de 512, que son los valores por defecto de `train_caser.py` salvo L (el original usa 5). Adam con las betas por defecto y el L2 como `weight_decay`, igual que el original. Además: un negativo uniforme por fila, hasta 200 épocas con early stopping por NDCG@10 (id) en `valid` con paciencia 10, y ranking sobre todo el catálogo de `info/`, igual que los métodos con LLM.

**Del original se usa solo la clase `Caser`** (`caser.py`, con `activation_getter` de `utils.py`), sin modificarla. En `caser_baseline.py`:
- `inicializar_proceso`: verifica el sha256 de `caser.py` y `utils.py` y los importa sin agregar la carpeta a `sys.path`. Como `caser.py` hace `from utils import ...`, `utils.py` se registra como módulo `utils` solo mientras se ejecuta `caser.py`.
- `preparar`: rellena los historiales con padding a la izquierda hasta 10, como `interactions.py`. El índice 0 queda reservado para el padding, con su propio embedding entrenable como en el original, por lo que dentro del modelo los ids van +1. Asigna a cada fila su índice de usuario (0 si no tiene filas en `train`) y arma los objetivos de la variante T=3.
- `nuevo_modelo`: crea el modelo con la inicialización de su constructor y pone en cero el usuario 0. Ese usuario nunca aparece en `train`, así que su gradiente es cero y sigue en cero.
- `convoluciones_como_matmul`: reemplaza el `forward` de las capas `conv_v` y `conv_h` de cada modelo por una multiplicación de matrices equivalente (`conv_v_matmul` y `conv_h_matmul`). `Caser.forward`, los parámetros y el `state_dict` son los del original. Se activa con `CONV_MATMUL`; el motivo y las mediciones están en `README.md` (Adaptaciones y Problemas conocidos).
- `puntuar`: usa `forward` del original para unos pocos ítems por fila (pérdidas BCE y T=3).
- `representacion` y `puntajes`: `representacion` toma de `forward` el vector x = [z, usuario] con hooks en `fc1` y en el embedding de usuario. `puntajes` calcula b2 + W2·x para todo el catálogo con una sola multiplicación, sin copiar W2 por fila, y salta el padding, así que la columna j es el id j.
- `perdida_bce`: la del paper (el objetivo contra un negativo uniforme). Es la pérdida del original con T=1 y un negativo.
- `perdida_ce`: referencia, softmax sobre el catálogo (con `puntajes`).
- `perdida_bce_t3`: referencia, 3 objetivos y 3 negativos por fila.
- `para_variante`: en "BCE sin usuario", todas las filas usan el usuario 0, fijo en cero.
- `entrenar`: Adam con weight decay, early stopping en `valid` y vuelta a la mejor época. Alterna los modos train y eval, porque el modelo tiene dropout.
- `correr_combinacion`: entrena una combinación con las semillas fijadas y devuelve su NDCG@10 en `valid`, sus métricas en `test` (también por usuarios conocidos y nuevos), el tiempo de entrenamiento y la memoria máxima de GPU.

## Prueba rápida

Entrena 3 épocas de una combinación de CDs con cada variante, después de una época de calentamiento que no se mide. Verifica:
- que el usuario 0 siga en cero;
- que `puntajes` dé lo mismo que `forward` del original para todo el catálogo, con la columna j como el id j;
- que las convoluciones como multiplicación de matrices den lo mismo que `nn.Conv2d`: un modelo sin el reemplazo, con el mismo estado, da los mismos puntajes y los mismos gradientes de la pérdida;
- que la evaluación sea determinista y que las cuatro pérdidas den valores finitos.

Muestra el tiempo por época y la memoria de GPU de cada variante, y estima el peor caso de la grilla con un proceso.

In [6]:
categoria, lr, wd, epocas = 'CDs_and_Vinyl', 1e-3, 1e-5, 3
datos, titulos, usuarios, prep = cb.cargar_categoria(base, categoria)
n_items = len(titulos)
prueba, modelos = {}, {}
for variante in cb.VARIANTES:
    p, n_usuarios = cb.para_variante(prep, usuarios, variante)
    args = (variante, lr, wd, p['train'], p['valid'], datos['valid'], n_items, n_usuarios)
    cb.entrenar(*args, max_epocas=1)   # calentamiento
    cb.fijar_semillas()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    inicio = time.time()
    modelo, _, _, corridas_prueba = cb.entrenar(*args, max_epocas=epocas)
    prueba[variante] = {'s por época': (time.time() - inicio) / corridas_prueba,
                        'MiB de GPU': torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else 0}
    modelos[variante] = (modelo, p)

    # El usuario 0 sigue en cero: no aparece en train, o está congelado en "BCE sin usuario"
    assert not modelo.user_embeddings.weight[0].any()
    # puntajes (multiplicación con W2 y b2) da lo mismo que forward del original para todo el catálogo, y
    # la columna j es el id j (en modo eval, sin dropout)
    modelo.eval()
    S, u = p['test']['S'][:8], p['test']['u'][:8]
    todos = torch.arange(1, n_items + 1, device=cb.device).expand(8, -1)
    with torch.no_grad():
        assert torch.allclose(cb.puntajes(modelo, S, u), cb.puntuar(modelo, S, u, todos), atol=1e-5)
    # Las convoluciones como multiplicación de matrices dan lo mismo que nn.Conv2d: un modelo sin el
    # reemplazo, con el mismo estado, da los mismos puntajes y los mismos gradientes (mismo dropout)
    cb.CONV_MATMUL = False
    referencia = cb.nuevo_modelo(n_items, n_usuarios, cb.VARIANTES[variante][1])
    cb.CONV_MATMUL = True
    referencia.load_state_dict(modelo.state_dict())
    referencia.eval()
    with torch.no_grad():
        assert torch.allclose(cb.puntajes(modelo, S, u), cb.puntajes(referencia, S, u), atol=1e-5)
    lote = next(cb.lotes(p['train'], cb.BATCH_SIZE, np.random.default_rng(0)))
    gradientes = []
    for m in [modelo, referencia]:
        m.train(); m.zero_grad(); torch.manual_seed(0)
        cb.VARIANTES[variante][0](m, lote).backward()
        gradientes.append([q.grad for q in m.parameters() if q.grad is not None])
    assert all((a - b).abs().max() <= 1e-4 * b.abs().max() for a, b in zip(*gradientes))
    modelo.zero_grad()
    modelo.eval()
    # En modo eval el modelo es determinista y las cuatro pérdidas son finitas
    recs = cb.recomendar(modelo, p['test'])
    assert recs == cb.recomendar(modelo, p['test'])
    lote = next(cb.lotes(p['train'], cb.BATCH_SIZE, np.random.default_rng(0)))
    with torch.no_grad():
        assert all(torch.isfinite(f(modelo, lote)) for f, _ in cb.VARIANTES.values())

# Peor caso: las 9 combinaciones de cada variante y dataset llegan a 200 épocas. El tiempo por época
# se escala con el número de filas de train. Es con un proceso; la grilla corre en 6 (README.md)
filas = {c: len(cb.cargar_categoria(base, c)[0]['train']) for c in names}
escala = sum(filas[c] / filas[categoria] for c in names)
for variante, r in prueba.items():
    r['peor caso con 1 proceso (h)'] = escala * r['s por época'] * cb.MAX_EPOCAS * 9 / 3600
print('Tiempo por época en CDs (incluye la evaluación en valid), memoria y peor caso de la grilla:')
display(pd.DataFrame(prueba).T.round(2))

modelo, p = modelos['BCE (paper)']
recs = cb.recomendar(modelo, p['test'])
metricas = {**cb.evaluar(recs, datos['test'], titulos),   # verifica que por título nunca sea menor que por id
            **cb.metricas_por_usuario(recs, datos['test'], titulos, usuarios)}
pd.Series(metricas, name='BCE (paper), 3 épocas').round(4)

MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/run/host/var/home/maurice/Documents/University/2026-2/Sistemas Recomendadores/Proyecto/iic3633-proyecto-latentr3/.venv-rocm/lib/python3.12/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1201_32.HIP.fdb.txt"


Tiempo por época en CDs (incluye la evaluación en valid), memoria y peor caso de la grilla:


,s por época,MiB de GPU,peor caso con 1 proceso (h)
BCE (paper),0.60,171.55,1.49
BCE sin usuario,0.49,185.29,1.21
CE catálogo,0.52,203.56,1.28
BCE T=3,0.53,206.17,1.33


HR@5 id                     0.0045
NDCG@5 id                   0.0031
HR@10 id                    0.0106
NDCG@10 id                  0.0050
HR@5 título                 0.0045
NDCG@5 título               0.0031
HR@10 título                0.0117
NDCG@10 título              0.0054
HR@10 título conocidos      0.0132
NDCG@10 título conocidos    0.0063
HR@10 título nuevos         0.0103
NDCG@10 título nuevos       0.0046
Name: BCE (paper), 3 épocas, dtype: float64

## Grilla y evaluación

Para cada variante y dataset se entrenan las 9 combinaciones de lr ∈ {1e-2, 1e-3, 1e-4} × weight decay ∈ {1e-4, 1e-5, 1e-6}. Es la misma grilla para todos los baselines tradicionales (Apéndice B). Las combinaciones se reparten en 6 procesos (`N_PROCESOS = 6`). Cada combinación fija sus semillas antes de empezar. Limitación: dos corridas de la grilla pueden diferir en las combinaciones con lr 1e-2, también en secuencia (ver `README.md`, Problemas conocidos).

Las métricas de `test` se calculan para todas las combinaciones, pero la configuración se elige **solo** con el mejor NDCG@10 (id) en `valid`. Ante un empate gana la primera en el orden de la grilla. La variante comparable con la Tabla 1 es **BCE (paper)** con el criterio **por título**.

In [7]:
LRS = [1e-2, 1e-3, 1e-4]
WEIGHT_DECAYS = [1e-4, 1e-5, 1e-6]
corridas = {}   # variante → DataFrame con sus 36 combinaciones, en el orden de la grilla

def correr_grilla(variante, n_procesos=N_PROCESOS):
    # Entrena las 36 combinaciones de una variante (4 datasets × 9) repartidas en n_procesos procesos
    combinaciones = list(itertools.product(names, LRS, WEIGHT_DECAYS))
    filas = [None] * len(combinaciones)
    inicio = time.time()
    contexto = multiprocessing.get_context('spawn')   # CUDA no admite procesos creados con fork
    with ProcessPoolExecutor(n_procesos, mp_context=contexto, initializer=cb.inicializar_proceso,
                             initargs=(codigo,)) as pool:
        futuros = {pool.submit(cb.correr_combinacion, base, categoria, variante, lr, wd): i
                   for i, (categoria, lr, wd) in enumerate(combinaciones)}
        for n, futuro in enumerate(as_completed(futuros), 1):
            r = futuro.result()
            filas[futuros[futuro]] = r
            print(f"[{n}/{len(combinaciones)}] {variante} {names[r['categoria']]} lr={r['lr']} wd={r['weight_decay']}: "
                  f"NDCG@10 valid {r['NDCG@10 valid']:.4f} (época {r['época']} de {r['épocas corridas']}) "
                  f"[{(time.time() - inicio) / 60:.1f} min]", flush=True)
    filas = pd.DataFrame(filas)
    filas.insert(1, 'dataset', filas.pop('categoria').map(names))
    return filas

### BCE (paper)

La variante del paper: el objetivo de la fila contra un negativo uniforme, con el embedding de usuario. Es la que se compara con la Tabla 1.

In [8]:
corridas['BCE (paper)'] = correr_grilla('BCE (paper)')

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


[1/36] BCE (paper) cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0170 (época 15 de 25) [7.8 min]
[2/36] BCE (paper) cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0094 (época 19 de 29) [7.9 min]
[3/36] BCE (paper) cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0082 (época 19 de 29) [7.9 min]
[4/36] BCE (paper) cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0066 (época 13 de 23) [8.2 min]
[5/36] BCE (paper) cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0068 (época 13 de 23) [8.3 min]
[6/36] BCE (paper) cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0068 (época 13 de 23) [8.4 min]
[7/36] BCE (paper) cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0112 (época 48 de 58) [8.5 min]
[8/36] BCE (paper) cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0147 (época 51 de 61) [8.7 min]
[9/36] BCE (paper) music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0678 (época 19 de 29) [9.0 min]
[10/36] BCE (paper) music_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0553 (época 14 de 24) [9.2 min]
[11/36] BCE (paper) music_data lr=0.01 wd=1e-05: 

### BCE sin usuario (referencia)

La misma BCE, con el vector de usuario fijo en cero en todas las filas: el puntaje depende solo de los ≤10 ítems del historial, igual que en GRU4Rec, SASRec y los LLM. Mide cuánto aporta el embedding de usuario y si el vector cero de los usuarios nuevos los perjudica.

In [9]:
corridas['BCE sin usuario'] = correr_grilla('BCE sin usuario')

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


[1/36] BCE sin usuario cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0068 (época 1 de 11) [0.4 min]
[2/36] BCE sin usuario cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0068 (época 1 de 11) [0.4 min]
[3/36] BCE sin usuario cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0068 (época 1 de 11) [0.4 min]
[4/36] BCE sin usuario cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0062 (época 2 de 12) [0.6 min]
[5/36] BCE sin usuario cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0204 (época 25 de 35) [0.9 min]
[6/36] BCE sin usuario cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0071 (época 25 de 35) [1.1 min]
[7/36] BCE sin usuario cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0302 (época 50 de 60) [1.2 min]
[8/36] BCE sin usuario cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0073 (época 34 de 44) [1.3 min]
[9/36] BCE sin usuario music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0703 (época 19 de 29) [1.4 min]
[10/36] BCE sin usuario music_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0571 (época 17 de 27) [1.9 min]
[11/36] BCE si

### CE catálogo (referencia)

Softmax sobre todo el catálogo. Mide cuánto le cuesta a Caser la BCE con un negativo. Con los puntajes como multiplicación de matrices usa poca memoria (ver `README.md`).

In [10]:
corridas['CE catálogo'] = correr_grilla('CE catálogo', n_procesos=N_PROCESOS_CE)

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


[1/36] CE catálogo cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0167 (época 3 de 13) [0.4 min]
[2/36] CE catálogo cd_data lr=0.001 wd=1e-06: NDCG@10 valid 0.0175 (época 10 de 20) [0.6 min]
[3/36] CE catálogo cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0171 (época 10 de 20) [0.6 min]
[4/36] CE catálogo cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0160 (época 13 de 23) [0.7 min]
[5/36] CE catálogo cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0060 (época 2 de 12) [0.7 min]
[6/36] CE catálogo cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0652 (época 21 de 31) [0.9 min]
[7/36] CE catálogo cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0061 (época 2 de 12) [0.9 min]
[8/36] CE catálogo cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0061 (época 2 de 12) [0.9 min]
[9/36] CE catálogo music_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0776 (época 2 de 12) [1.1 min]
[10/36] CE catálogo music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.1056 (época 6 de 16) [1.2 min]
[11/36] CE catálogo music_data lr=0.01 wd=1e-06: NDCG@10 v

### BCE T=3 (referencia)

Cada fila supervisa su objetivo y los 2 ítems siguientes del mismo usuario que estén en `train`, contra 3 negativos uniformes, con la pérdida del original. Mide cuánto le cuesta a Caser entrenar con un solo objetivo por fila ([decisiones comunes](../README.md#decisiones-comunes-d1d17)).

In [11]:
corridas['BCE T=3'] = correr_grilla('BCE T=3')

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48
E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


[1/36] BCE T=3 cd_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0135 (época 11 de 21) [0.6 min]
[2/36] BCE T=3 cd_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0100 (época 14 de 24) [0.7 min]
[3/36] BCE T=3 cd_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0190 (época 28 de 38) [1.1 min]
[4/36] BCE T=3 cd_data lr=0.0001 wd=0.0001: NDCG@10 valid 0.0069 (época 12 de 22) [1.1 min]
[5/36] BCE T=3 cd_data lr=0.001 wd=1e-05: NDCG@10 valid 0.0149 (época 33 de 43) [1.2 min]
[6/36] BCE T=3 cd_data lr=0.0001 wd=1e-05: NDCG@10 valid 0.0073 (época 13 de 23) [1.2 min]
[7/36] BCE T=3 cd_data lr=0.0001 wd=1e-06: NDCG@10 valid 0.0073 (época 13 de 23) [1.6 min]
[8/36] BCE T=3 cd_data lr=0.001 wd=0.0001: NDCG@10 valid 0.0204 (época 53 de 63) [1.6 min]
[9/36] BCE T=3 music_data lr=0.01 wd=0.0001: NDCG@10 valid 0.0665 (época 11 de 21) [1.8 min]
[10/36] BCE T=3 music_data lr=0.01 wd=1e-06: NDCG@10 valid 0.0547 (época 11 de 21) [1.9 min]
[11/36] BCE T=3 music_data lr=0.01 wd=1e-05: NDCG@10 valid 0.0644 (época 16 de 26) [2.0 min

### Mejor configuración por variante y dataset

Usa las variantes que se hayan corrido, y elige solo con `valid`.

In [12]:
print('Variantes corridas:', list(corridas))
todas = pd.concat(corridas.values(), ignore_index=True)
# Mejor configuración por variante y dataset, elegida solo con valid
resultados = todas.loc[todas.groupby(['variante', 'dataset'], sort=False)['NDCG@10 valid'].idxmax()]
resultados = resultados.reset_index(drop=True)
resultados.round(4)

Variantes corridas: ['BCE (paper)', 'BCE sin usuario', 'CE catálogo', 'BCE T=3']


,variante,dataset,lr,weight_decay,época,épocas corridas,segundos,MiB de GPU,NDCG@10 valid,HR@5 id,...,HR@10 id,NDCG@10 id,HR@5 título,NDCG@5 título,HR@10 título,NDCG@10 título,HR@10 título conocidos,NDCG@10 título conocidos,HR@10 título nuevos,NDCG@10 título nuevos
0,BCE (paper),cd_data,0.001,0.0000,163,173,649.9835,143.9951,0.0260,0.0297,...,0.0414,0.0268,0.0328,0.0250,0.0465,0.0294,0.0451,0.0338,0.0478,0.0252
1,BCE (paper),music_data,0.010,0.0000,32,42,62.9101,147.3164,0.0795,0.0754,...,0.0986,0.0685,0.0754,0.0610,0.0988,0.0685,0.0626,0.0402,0.1464,0.1058
2,BCE (paper),toys_data,0.001,0.0000,151,161,169.9579,168.7759,0.0350,0.0324,...,0.0435,0.0264,0.0324,0.0228,0.0435,0.0264,0.0202,0.0113,0.0736,0.0460
3,BCE (paper),games_data,0.010,0.0000,10,20,37.8689,170.7090,0.0324,0.0276,...,0.0441,0.0230,0.0276,0.0177,0.0441,0.0230,0.0404,0.0212,0.0475,0.0248
4,BCE sin usuario,cd_data,0.010,0.0000,89,99,125.0689,136.4438,0.0314,0.0387,...,0.0516,0.0324,0.0406,0.0295,0.0567,0.0346,0.0447,0.0310,0.0683,0.0382
5,BCE sin usuario,music_data,0.001,0.0001,63,73,105.8472,136.6606,0.0840,0.0796,...,0.0971,0.0745,0.0796,0.0689,0.0971,0.0745,0.0618,0.0444,0.1436,0.1141
6,BCE sin usuario,toys_data,0.010,0.0000,38,48,61.6656,159.6787,0.0274,0.0191,...,0.0285,0.0158,0.0191,0.0128,0.0285,0.0158,0.0155,0.0077,0.0453,0.0263
7,BCE sin usuario,games_data,0.010,0.0000,8,18,31.6940,160.4336,0.0335,0.0235,...,0.0399,0.0210,0.0237,0.0158,0.0403,0.0212,0.0351,0.0191,0.0452,0.0231
8,CE catálogo,cd_data,0.010,0.0001,21,31,49.4709,153.7212,0.0652,0.0773,...,0.0926,0.0680,0.0823,0.0661,0.1013,0.0721,0.0816,0.0598,0.1206,0.0842
9,CE catálogo,music_data,0.010,0.0001,6,16,29.1574,157.8467,0.1056,0.0956,...,0.1175,0.0896,0.0956,0.0826,0.1175,0.0896,0.0762,0.0541,0.1721,0.1364


## Comparación con la Tabla 1 del paper

Caser según el paper, junto a la variante BCE por id y por título y las variantes de referencia por título. Las variantes de referencia no se comparan con la Tabla 1: muestran cuánto cambia Caser sin el embedding de usuario, con otra pérdida (CE) o con la supervisión del original (T=3).

In [13]:
PAPER = {'toys_data': [0.0251, 0.0384, 0.0170, 0.0214], 'cd_data': [0.0469, 0.0689, 0.0312, 0.0382],
         'games_data': [0.0324, 0.0538, 0.0211, 0.0280], 'music_data': [0.0781, 0.0977, 0.0564, 0.0627]}
METRICAS = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']
FUENTES = [('BCE (paper), id', 'BCE (paper)', 'id'), ('BCE (paper), título', 'BCE (paper)', 'título'),
           ('BCE sin usuario, título', 'BCE sin usuario', 'título'),
           ('CE catálogo, título', 'CE catálogo', 'título'), ('BCE T=3, título', 'BCE T=3', 'título')]

def fila(variante, nombre, criterio):
    r = resultados[(resultados['variante'] == variante) & (resultados['dataset'] == nombre)].iloc[0]
    return [r[f'{m} {criterio}'] for m in METRICAS]

tabla = []
for nombre in names.values():
    tabla.append({'dataset': nombre, 'fuente': 'Paper', **dict(zip(METRICAS, PAPER[nombre]))})
    for fuente, variante, criterio in FUENTES:
        if variante in corridas:   # solo las variantes que se corrieron
            tabla.append({'dataset': nombre, 'fuente': fuente,
                          **dict(zip(METRICAS, fila(variante, nombre, criterio)))})
pd.DataFrame(tabla).round(4)

,dataset,fuente,HR@5,HR@10,NDCG@5,NDCG@10
0,cd_data,Paper,0.0469,0.0689,0.0312,0.0382
1,cd_data,"BCE (paper), id",0.0297,0.0414,0.0230,0.0268
2,cd_data,"BCE (paper), título",0.0328,0.0465,0.0250,0.0294
3,cd_data,"BCE sin usuario, título",0.0406,0.0567,0.0295,0.0346
4,cd_data,"CE catálogo, título",0.0823,0.1013,0.0661,0.0721
5,cd_data,"BCE T=3, título",0.0307,0.0486,0.0228,0.0285
6,music_data,Paper,0.0781,0.0977,0.0564,0.0627
7,music_data,"BCE (paper), id",0.0754,0.0986,0.0610,0.0685
8,music_data,"BCE (paper), título",0.0754,0.0988,0.0610,0.0685
9,music_data,"BCE sin usuario, título",0.0796,0.0971,0.0689,0.0745


## Usuarios conocidos y nuevos

HR@10 y NDCG@10 por título en `test`, por separado para las filas de usuarios con filas en `train` y sin ellas, con la configuración elegida de cada variante. Los usuarios nuevos son el 43–52 % de las filas de test. Si con el embedding de usuario los nuevos quedan muy por debajo de "BCE sin usuario", el vector cero los perjudica.

In [14]:
columnas = ['HR@10 título conocidos', 'NDCG@10 título conocidos', 'HR@10 título nuevos', 'NDCG@10 título nuevos']
resultados.set_index(['dataset', 'variante'])[columnas].sort_index().round(4)

HR@10 título conocidos  NDCG@10 título conocidos  \
dataset    variante                                                            
cd_data    BCE (paper)                      0.0451                    0.0338   
           BCE T=3                          0.0493                    0.0326   
           BCE sin usuario                  0.0447                    0.0310   
           CE catálogo                      0.0816                    0.0598   
games_data BCE (paper)                      0.0404                    0.0212   
           BCE T=3                          0.0419                    0.0217   
           BCE sin usuario                  0.0351                    0.0191   
           CE catálogo                      0.0564                    0.0318   
music_data BCE (paper)                      0.0626                    0.0402   
           BCE T=3                          0.0535                    0.0334   
           BCE sin usuario                  0.0618                    0.0444   
           CE catálogo                      0.0762                    0.0541   
toys_data  BCE (paper)                      0.0202                    0.0113   
           BCE T=3                          0.0221                    0.0108   
           BCE sin usuario                  0.0155                    0.0077   
           CE catálogo                      0.0379                    0.0233   

                            HR@10 título nuevos  NDCG@10 título nuevos  
dataset    variante                                                     
cd_data    BCE (paper)                   0.0478                 0.0252  
           BCE T=3                       0.0478                 0.0244  
           BCE sin usuario               0.0683                 0.0382  
           CE catálogo                   0.1206                 0.0842  
games_data BCE (paper)                   0.0475                 0.0248  
           BCE T=3                       0.0661                 0.0347  
           BCE sin usuario               0.0452                 0.0231  
           CE catálogo                   0.1074                 0.0625  
music_data BCE (paper)                   0.1464                 0.1058  
           BCE T=3                       0.1260                 0.0902  
           BCE sin usuario               0.1436                 0.1141  
           CE catálogo                   0.1721                 0.1364  
toys_data  BCE (paper)                   0.0736                 0.0460  
           BCE T=3                       0.0518                 0.0297  
           BCE sin usuario               0.0453                 0.0263  
           CE catálogo                   0.1257                 0.0886

## Pisos de referencia

MostPop por frecuencia en train (HR@10 y NDCG@10 en test, definidos en `../README.md`, sección de pisos). Si la variante BCE queda por debajo, probablemente hay un error.

In [15]:
PISOS = {'cd_data': (0.0197, 0.0091), 'music_data': (0.0659, 0.0330),
         'toys_data': (0.0138, 0.0082), 'games_data': (0.0208, 0.0111)}
assert 'BCE (paper)' in corridas, 'Falta correr la celda de la variante BCE (paper)'
for nombre, (piso_hr, piso_ndcg) in PISOS.items():
    r = resultados[(resultados['variante'] == 'BCE (paper)') & (resultados['dataset'] == nombre)].iloc[0]
    ok = r['HR@10 id'] >= piso_hr and r['NDCG@10 id'] >= piso_ndcg
    print(f"{nombre}: HR@10 id {r['HR@10 id']:.4f} (piso {piso_hr}), "
          f"NDCG@10 id {r['NDCG@10 id']:.4f} (piso {piso_ndcg}): {'OK' if ok else 'ALERTA'}")

cd_data: HR@10 id 0.0414 (piso 0.0197), NDCG@10 id 0.0268 (piso 0.0091): OK
music_data: HR@10 id 0.0986 (piso 0.0659), NDCG@10 id 0.0685 (piso 0.033): OK
toys_data: HR@10 id 0.0435 (piso 0.0138), NDCG@10 id 0.0264 (piso 0.0082): OK
games_data: HR@10 id 0.0441 (piso 0.0208), NDCG@10 id 0.0230 (piso 0.0111): OK
